# Day 5: Structured Output — Getting JSON Back Reliably

**30-Day AI Agent Challenge: OpenAI Agents SDK vs LangGraph vs CrewAI**

---

## Today's Concept

> Free-text responses are great for chatting. Structured responses are great for building.

When you're building real applications, you need the agent to return data in a
**predictable format** — not a paragraph, but a JSON object with specific fields.
Today we ask each framework to return a structured `CountryInfo` response.

## What You'll Build
- Define a `CountryInfo` schema (capital, population, continent, currency)
- Ask each framework to return data matching this schema
- See which framework enforces structure most reliably

In [1]:
# ── Setup ───────────────────────────────────────────────
import sys
sys.path.insert(0, "..")
from shared import check_and_report, print_config, disable_openai_tracing, get_openai_agents_model, get_langgraph_model, get_crewai_llm
check_and_report()
disable_openai_tracing()
print("=" * 50)
print("DAY 5 SETUP COMPLETE")
print("=" * 50)
print_config()

Python: 3.12.13

All required packages installed.
Optional (missing): langchain-openai, langgraph-checkpoint-memory, ipywidgets

Ollama: connected (8 model(s) available)

DAY 5 SETUP COMPLETE
  Provider: OLLAMA
  Model:    qwen2.5:3b
  Ollama:   http://localhost:11434


## Today's Question

> **"Tell me about Japan."**

But instead of free text, we want a structured response:
```json
{
  "country": "Japan",
  "capital": "Tokyo",
  "continent": "Asia",
  "currency": "Japanese Yen",
  "fun_fact": "..."
}
```

---
## Framework 1: OpenAI Agents SDK

Pass a Pydantic model as `output_type` on the Agent. The SDK handles the rest.

In [2]:
from agents import Agent, Runner
from pydantic import BaseModel

model = get_openai_agents_model()

# ── Define the output schema ─────────────────────────────
class CountryInfo(BaseModel):
    country: str
    capital: str
    continent: str
    currency: str
    fun_fact: str

# ── Agent with structured output ────────────────────────
agent = Agent(
    name="Country Expert",
    instructions="You are a geography expert. Always return structured country information.",
    model=model,
    output_type=CountryInfo,  # THIS enforces the schema
)

result = result = await Runner.run(agent, "Tell me about Japan.")

print("=" * 60)
print("OPENAI AGENTS SDK — STRUCTURED OUTPUT")
print("=" * 60)
# result.final_output IS the Pydantic model
info = result.final_output
print(f"Type: {type(info).__name__}")
print(f"Country:  {info.country}")
print(f"Capital:   {info.capital}")
print(f"Continent: {info.continent}")
print(f"Currency:  {info.currency}")
print(f"Fun Fact:  {info.fun_fact}")
print(f"\nFull JSON:")
print(info.model_dump_json(indent=2))

OPENAI AGENTS SDK — STRUCTURED OUTPUT
Type: CountryInfo
Country:  Japan
Capital:   Tokyo
Continent: Asia
Currency:  Japanese Yen (JPY)
Fun Fact:  Mount Fuji is the highest mountain in Japan and is an iconic symbol of the country, being an active stratovolcano. It has been a UNESCO World Heritage Site since 2013, with its status divided into natural and cultural components as it also represents significant religious landscapes including Shinto shrines at its base and Buddhist temples in its upper slopes.

Full JSON:
{
  "country": "Japan",
  "capital": "Tokyo",
  "continent": "Asia",
  "currency": "Japanese Yen (JPY)",
  "fun_fact": "Mount Fuji is the highest mountain in Japan and is an iconic symbol of the country, being an active stratovolcano. It has been a UNESCO World Heritage Site since 2013, with its status divided into natural and cultural components as it also represents significant religious landscapes including Shinto shrines at its base and Buddhist temples in its upper slop

---
## Framework 2: LangGraph

LangGraph uses `with_structured_output()` on the model or a dedicated parsing node.

In [3]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel
import json

model = get_langgraph_model()

# ── Define the output schema ─────────────────────────────
class CountryInfo(BaseModel):
    country: str
    capital: str
    continent: str
    currency: str
    fun_fact: str

# ── Use with_structured_output on the model ───────────────
structured_model = model.with_structured_output(CountryInfo)

# For structured output, a direct LLM call is cleaner than create_react_agent
result = structured_model.invoke([
    SystemMessage(content="You are a geography expert. Return structured country information."),
    HumanMessage(content="Tell me about Japan."),
])

print("=" * 60)
print("LANGGRAPH — STRUCTURED OUTPUT")
print("=" * 60)
print(f"Type: {type(result).__name__}")
print(f"Country:  {result.country}")
print(f"Capital:   {result.capital}")
print(f"Continent: {result.continent}")
print(f"Currency:  {result.currency}")
print(f"Fun Fact:  {result.fun_fact}")
print(f"\nFull JSON:")
print(result.model_dump_json(indent=2))

LANGGRAPH — STRUCTURED OUTPUT
Type: CountryInfo
Country:  Japan
Capital:   Tokyo
Continent: Asia
Currency:  Japanese yen (JPY)
Fun Fact:  Over 73% of Japanese households own at least one pet, the highest in the world per capita and significantly higher than any other country's percentage reported by the World Organization for Animal Health (OIE). In Japan, dog ownership is even more prevalent, with about 2.4 dogs for every 10 people, the highest dog ownership rate globally among both urban and rural populations, according to data from OIE as of 2023. This highlights an interesting contrast in pet ownership trends compared to countries like the United States where cat ownership is much more prevalent than dog ownership per capita. In Japan, dogs often wear clothes similar to humans, leading to a trend known as 

Full JSON:
{
  "country": "Japan",
  "capital": "Tokyo",
  "continent": "Asia",
  "currency": "Japanese yen (JPY)",
  "fun_fact": "Over 73% of Japanese households own at least o

---
## Framework 3: CrewAI

CrewAI uses `output_json` on the Task to enforce structured output.

In [9]:
from crewai import Agent, Task, Crew, Process
from pydantic import BaseModel
import json

llm = get_crewai_llm()

class CountryInfo(BaseModel):
    country: str
    capital: str
    continent: str
    currency: str
    fun_fact: str

expert = Agent(
    role="Country Expert",
    goal="Provide structured country information",
    backstory="You are a meticulous geographer who always organizes information clearly.",
    llm=llm,
    verbose=True,
)

task = Task(
    description="Tell me about Japan. Return a JSON object with country, capital, continent, currency, and fun_fact.",
    expected_output="A JSON object with country, capital, continent, currency, fun_fact.",
    agent=expert,
    output_json=CountryInfo,
)

crew = Crew(agents=[expert], tasks=[task], process=Process.sequential, verbose=True)
result = await crew.kickoff_async()

print()
print("=" * 60)
print("CREWAI — STRUCTURED OUTPUT")
print("=" * 60)

if result.json_dict:
    info = result.json_dict
    print(f"Country:   {info['country']}")
    print(f"Capital:   {info['capital']}")
    print(f"Continent: {info['continent']}")
    print(f"Currency:  {info['currency']}")
    print(f"Fun Fact:  {info['fun_fact']}")
    print("\nFull JSON:")
    print(json.dumps(info, indent=2))
else:
    print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 4fd3c053-e4cc-453f-a9e8-ec7b2e9ab13d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Tell me about Japan. Return a JSON object with country, capital, continent, currency, and fun_fact.      │
│  ID: 9ed18455-d062-4bcd-bcc7-7cbfcdb42317                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Country Expert                                                                                          │
│                                                                                                                 │
│  Task: Tell me about Japan. Return a JSON object with country, capital, continent, currency, and fun_fact.      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Country Expert                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  country='Japan' capital='Tokyo' continent='Asia' currency='Japanese yen (¥)' fun_fact="The world's first       │
│  public advertisement was for an inn in Edo-era Japan, which advertised a ‘very delicious food’ in Chinese      │
│  characters."                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Tell me about Japan. Return a JSON object with country, capital, continent, currency, and fun_fact.      │
│  Agent: Country Expert                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 4fd3c053-e4cc-453f-a9e8-ec7b2e9ab13d                                                                       │
│  Final Output: {"country":"Japan","capital":"Tokyo","continent":"Asia","currency":"Japanese yen                 │
│  (¥)","fun_fact":"The world's first public advertisement was for an inn in Edo-era Japan, which advertised a    │
│  ‘very delicious food’ in Chinese characters."}                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


CREWAI — STRUCTURED OUTPUT
Country:   Japan
Capital:   Tokyo
Continent: Asia
Currency:  Japanese yen (¥)
Fun Fact:  The world's first public advertisement was for an inn in Edo-era Japan, which advertised a ‘very delicious food’ in Chinese characters.

Full JSON:
{
  "country": "Japan",
  "capital": "Tokyo",
  "continent": "Asia",
  "currency": "Japanese yen (\u00a5)",
  "fun_fact": "The world's first public advertisement was for an inn in Edo-era Japan, which advertised a \u2018very delicious food\u2019 in Chinese characters."
}


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## Comparison: Structured Output

| Framework | How to enforce | Returns | Notes |
|---|---|---|---|
| OpenAI SDK | `output_type=PydanticModel` on Agent | Pydantic model | Cleanest API |
| LangGraph | `model.with_structured_output(Schema)` | Pydantic model | Most flexible (use on any model call) |
| CrewAI | `output_json=PydanticModel` on Task | Pydantic model | Tied to Task level |

**Key insight:** All three use Pydantic models as the schema definition. The difference
is WHERE you attach the schema — Agent, model call, or Task.

## Key Takeaway

Structured output turns agent responses from "string" to "object". This is what
lets you build real applications on top of agents. The schema is always a Pydantic
model — the frameworks differ only in where you attach it.

---

